# Lab 05 · MPI Primer · ranks, sends, receives, collectives

OpenMP scales to one node. To go beyond one node you need **MPI** — the message-passing model where processes on different machines talk to each other by explicitly sending and receiving messages. This lab is the primer: how ranks identify themselves, how to send/receive, what collectives buy you. Lab 06 puts MPI on the heat stencil.

**Prerequisites.** Lab 03 (OpenMP; you understand parallel work). No prior MPI required.

**Builds toward.** Lab 06 (2D domain decomposition of the heat stencil). Lab 07 (hybrid).

> **📚 Where to look when you're stuck**
>
> - [**MPI 4.1 standard**](https://www.mpi-forum.org/docs/) — authoritative reference
> - [**LLNL MPI tutorial**](https://hpc-tutorials.llnl.gov/mpi/) — best beginner walkthrough
> - [**Crux MPI + compilers**](https://docs.alcf.anl.gov/crux/) — which MPI to load



## How this notebook works

Same three surfaces as lab 01 and 02: **[Hub]**, **[Hub -> Crux]**, **[Crux compute]**.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab05", host="crux",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + prerequisite artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab05 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> Crux] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab05 dir ready')


## Part 1 · Hello from every rank

Every MPI program starts with `MPI_Init(&argc, &argv)` and ends with `MPI_Finalize()`. Between those, every process knows two things: its **rank** (0-based integer id) and the **total number of ranks** (world size).

Ranks 0 through N-1 are logically distinct processes even if they're on the same node. They see each other only through MPI calls.


In [ ]:
# [Hub] Write mpiHello.c.
(labDir/'mpiHello.c').write_text('''
#include <mpi.h>
#include <stdio.h>
int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    char host[256]; gethostname(host, 256);
    printf("rank %d of %d on %s\\n", rank, size, host);
    MPI_Finalize();
    return 0;
}
''')
showFile(labDir/'mpiHello.c', language='c', title='mpiHello.c')


In [ ]:
# [Hub -> Crux] Build + run on 2 nodes, 4 ranks per node.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
module load PrgEnv-cray 2>/dev/null || module load PrgEnv-gnu 2>/dev/null || true
cc -o mpiHello mpiHello.c
NNODES=$(wc -l < $PBS_NODEFILE); NRANKS_PER_NODE=4; NRANKS=$(( NNODES * NRANKS_PER_NODE ))
mpiexec -n $NRANKS --ppn $NRANKS_PER_NODE ./mpiHello | sort
'''
pbsPath = labDir/'helloJob.pbs'
pbsPath.write_text(pbsHeader(name='lab05Hello', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             select='2:system=crux', walltime='00:10:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/hello.out') + jobBody)
sshPut(str(labDir/'mpiHello.c'), env['HPC_LAB_DIR']+'/mpiHello.c')
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/helloJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/helloJob.pbs'); waitJob(jobID, 15, 900)
sshGet(env['HPC_LAB_DIR']+'/hello.out', str(labDir/'hello.out'))
print((labDir/'hello.out').read_text())


In [ ]:
checkpoint("Part 1 - mpi hello", [
    check("hello output has 8 ranks",
          lambda: (sum(1 for l in open(labDir/'hello.out') if 'rank' in l) >= 8, 'ok')),
])


## Part 2 · Point-to-point · `MPI_Send` and `MPI_Recv`

The primitive of message passing is a **matched send/receive pair**. One rank calls `MPI_Send(buf, count, type, dest, tag, comm)`; another calls `MPI_Recv(buf, count, type, source, tag, comm, &status)`. The runtime moves the bytes.

The signatures are notorious for their length; the parts you'll change every time are `buf`, `count`, and `dest` / `source`. Everything else is boilerplate.


In [ ]:
# [Hub] A ring-passing demo: rank r sends an int to rank (r+1)%size.
(labDir/'ring.c').write_text('''
#include <mpi.h>
#include <stdio.h>
int main(int argc, char **argv){
    MPI_Init(&argc,&argv);
    int r,s; MPI_Comm_rank(MPI_COMM_WORLD,&r); MPI_Comm_size(MPI_COMM_WORLD,&s);
    int recv, send=r*10;
    MPI_Sendrecv(&send,1,MPI_INT,(r+1)%s,0,
                 &recv,1,MPI_INT,(r-1+s)%s,0,
                 MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    printf("rank %d sent %d, received %d\\n", r, send, recv);
    MPI_Finalize(); return 0;
}
''')


In [ ]:
# [Hub -> Crux] Run the ring.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cc -o ring ring.c
mpiexec -n 8 ./ring | sort
'''
pbsPath = labDir/'ringJob.pbs'
pbsPath.write_text(pbsHeader(name='lab05Ring', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             walltime='00:05:00', filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/ring.out') + jobBody)
sshPut(str(labDir/'ring.c'), env['HPC_LAB_DIR']+'/ring.c')
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/ringJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/ringJob.pbs'); waitJob(jobID, 15, 600)
sshGet(env['HPC_LAB_DIR']+'/ring.out', str(labDir/'ring.out'))
print((labDir/'ring.out').read_text())


In [ ]:
checkpoint("Part 2 - ring send/recv", [
    check("ring output present", fileExists(str(labDir/'ring.out'))),
])


## Part 3 · Collectives · `MPI_Bcast`, `MPI_Reduce`, `MPI_Allreduce`

Collectives are single MPI calls that involve ALL ranks in a communicator. They replace hand-rolled trees of sends and receives:

| Call | What it does |
|---|---|
| `MPI_Bcast` | rank R sends the same data to every other rank |
| `MPI_Reduce` | every rank contributes; one rank gets the sum/max/min/… |
| `MPI_Allreduce` | every rank contributes; every rank gets the result |
| `MPI_Gather` | every rank sends; one rank collects the concatenated array |
| `MPI_Allgather` | every rank sends; every rank gets the concatenated array |
| `MPI_Barrier` | synchronize; nobody proceeds until everybody arrives |

**Use collectives whenever you can.** They're implemented with optimal algorithms (trees, pipelines) that a hand-rolled loop of Sends won't match.


In [ ]:
# [Hub] Simple reduction: sum ranks' contributions.
(labDir/'reduce.c').write_text('''
#include <mpi.h>
#include <stdio.h>
int main(int argc, char **argv){
    MPI_Init(&argc,&argv);
    int r,s; MPI_Comm_rank(MPI_COMM_WORLD,&r); MPI_Comm_size(MPI_COMM_WORLD,&s);
    int local = r + 1;
    int global;
    MPI_Allreduce(&local, &global, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
    if (r==0) printf("sum 1..%d = %d (should be %d)\\n", s, global, s*(s+1)/2);
    MPI_Finalize(); return 0;
}
''')


In [ ]:
# [Hub -> Crux] Build + run the reduction.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cc -o reduce reduce.c
mpiexec -n 16 ./reduce
'''
pbsPath = labDir/'redJob.pbs'
pbsPath.write_text(pbsHeader(name='lab05Red', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             walltime='00:05:00', filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/red.out') + jobBody)
sshPut(str(labDir/'reduce.c'), env['HPC_LAB_DIR']+'/reduce.c')
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/redJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/redJob.pbs'); waitJob(jobID, 15, 600)
sshGet(env['HPC_LAB_DIR']+'/red.out', str(labDir/'red.out'))
print((labDir/'red.out').read_text())


In [ ]:
checkpoint("Part 3 - collectives", [
    check("reduction ran", fileExists(str(labDir/'red.out'))),
])


## Part 4 · Non-blocking · `MPI_Isend` and `MPI_Irecv`

`MPI_Send` and `MPI_Recv` block until the operation is complete. **Non-blocking** versions (`MPI_Isend`, `MPI_Irecv`) return immediately with a request handle; you check completion later with `MPI_Wait` or `MPI_Test`.

Non-blocking calls are what let you **overlap computation and communication** — issue the message send, do interior work while it flies, then wait for the boundary data to arrive. This is the core trick in lab 06's halo exchange.


In [ ]:
# [Hub] Show the pattern; no separate run needed - lab 06 does the real thing.
showNote('Pattern used in lab 06:\n\n'
         '    MPI_Isend(boundary, ..., &reqSend);\n'
         '    MPI_Irecv(halo,     ..., &reqRecv);\n'
         '    computeInterior();               // overlaps with comm\n'
         '    MPI_Waitall(2, requests, statuses);\n'
         '    computeBoundary();               // uses halo data',
         kind='info')


In [ ]:
checkpoint("Part 4 - non-blocking pattern read", [
    check("labEnv.sh exists", fileExists(str(labDir/'labEnv.sh'))),
])


## Part 5 · Communicators · groups of ranks that talk

`MPI_COMM_WORLD` is the default: every rank in the job. You can create **sub-communicators** with `MPI_Comm_split` to talk to a subset — every row of a 2D grid, every socket-local group, every subset that shares a task.

Lab 06 will split MPI_COMM_WORLD into a 2D grid so the heat stencil's halo exchange talks only to the four neighbors on the Cartesian grid.


In [ ]:
checkpoint("Part 5 - communicators", [
    check("lab dir persistent", dirExists(str(labDir))),
])


## Part 6 · Bridge to lab 06

You now know enough MPI to write a real distributed program. Lab 06 does it: **2D domain decomposition** of the heat stencil, with a **halo exchange** between neighboring ranks, using the non-blocking pattern from Part 4 to overlap communication with interior computation.


## Wrap up

Moved the spine forward one lab. Ready for the next.


### Lab scorecard


In [ ]:
labSummary("MPI Primer")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("MPI Primer")
